# Reproducing Thesis Figures

This notebook reproduces every publication figure from the thesis.
All figures use `paper.mplstyle` for typographic consistency with the PDF document.

## Figure index

| Figure | Section below | Script equivalent |
|--------|--------------|-------------------|
| MAPE histogram | §3 | `replot_batch_results.py` |
| Prominent peaks comparison | §4 | `compare_model_versions.py` |
| Model comparison (Baseline vs Q-Weighted) | §5 | `compare_model_versions.py` |
| Coverage | §6 | `plot_mape_vs_std.py` |
| Single reflectivity curve fit | §7 | `simple_pipeline.py` |
| Evaluation against random guessing | §8 | `evaluate_random_guessing.py` |

---

> **Prerequisites**: Completed batch runs are expected in `paper_batches/` or `batch_inference_results/`.
> Run the sweeps first if needed — see the README for sweep commands.

## 1. Setup

In [ ]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

from config import DATA_DIRECTORY, BATCH_RESULTS_DIR, PAPER_BATCHES_DIR

# Output directory for figures generated in this notebook
FIGURES_DIR = Path(PROJECT_ROOT) / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Figures will be saved to: {FIGURES_DIR}")

## 2. Configure Batch Directories

Set the batch directory references used throughout this notebook.
These should point to completed runs in `paper_batches/` or `batch_inference_results/`.

In [ ]:
PAPER_BATCHES = Path(PROJECT_ROOT) / PAPER_BATCHES_DIR
BATCH_RESULTS = Path(PROJECT_ROOT) / BATCH_RESULTS_DIR

# Baseline model batch (all exps, no prominent filter, 30% constraint, no SLD fix)
BASELINE_ALL_DIR  = BATCH_RESULTS / "069_3169exps_1layers_30constraint_28october2025_14_41"

# Q-weighted model batch — update to the actual completed batch directory
QWEIGHTED_ALL_DIR = BASELINE_ALL_DIR   # replace with actual q-weighted batch dir

# Mean-conditioned model batch — update when available
MEAN_ALL_DIR      = BASELINE_ALL_DIR   # replace with actual mean-conditioned batch dir

# Single experiment for the curve-fit figure (choose one with a prominent oscillation peak)
SINGLE_EXP_ID = "s000780"

def _load_successful(batch_dir: Path):
    results_file = batch_dir / "batch_results.json"
    with open(results_file) as f:
        data = json.load(f)
    return {k: v for k, v in data.items() if v.get("success", False)}

print("Batch directories configured.")

## 3. Figure: MAPE Histograms (per-model)

Generates `mape_distribution_1layer.pdf` and `parameter_breakdown_1layer.pdf` for each model variant.

**Configuration**: 30% constraint prior, no SLD fixing (`--fix-sld-mode none`), no prominent-feature filtering.

Each run processes the full test split and takes approximately **15 minutes** on a modern GPU.

**CLI commands to produce the three model batches**:

```bash
# NF Baseline
python batch_pipeline.py --data-directory dataset/test \
  --priors-type constraint_based --priors-deviation 30 \
  --fix-sld-mode none --inference-backend nf \
  --config-name example_nf_config_reflectorch.yaml --nf-num-samples 1000

# NF + Q-Weighted + dR
python batch_pipeline.py --data-directory dataset/test \
  --priors-type constraint_based --priors-deviation 30 \
  --fix-sld-mode none --inference-backend nf \
  --config-name nf_config_mixed_sigmas_qweighted.yaml --nf-num-samples 1000 --use-sigmas-input

# NF + Mean Conditioned
python batch_pipeline.py --data-directory dataset/test \
  --priors-type constraint_based --priors-deviation 30 \
  --fix-sld-mode none --inference-backend nf \
  --config-name nf_config_mixed_mean_conditioned.yaml --nf-num-samples 1000
```

Update `BASELINE_ALL_DIR`, `QWEIGHTED_ALL_DIR`, and `MEAN_ALL_DIR` above, then re-run the cell below.

In [ ]:
from plotting_utils import create_batch_analysis_plots

# Baseline
baseline_results = _load_successful(BASELINE_ALL_DIR)
print(f"Baseline: {len(baseline_results)} successful experiments")
create_batch_analysis_plots(baseline_results, layer_count=1, output_dir=FIGURES_DIR / "baseline", save=True)

# Q-Weighted
qw_results = _load_successful(QWEIGHTED_ALL_DIR)
print(f"Q-Weighted: {len(qw_results)} successful experiments")
create_batch_analysis_plots(qw_results, layer_count=1, output_dir=FIGURES_DIR / "qweighted", save=True)

# Mean Conditioned
mean_results = _load_successful(MEAN_ALL_DIR)
print(f"Mean Conditioned: {len(mean_results)} successful experiments")
create_batch_analysis_plots(mean_results, layer_count=1, output_dir=FIGURES_DIR / "mean", save=True)

print("MAPE histogram figures saved.")

## 4. Figure: Single Reflectivity Curve with Highlighted Peak

Shows the experimental reflectivity curve overlaid with the MAP posterior prediction and
uncertainty band, plus the inferred SLD depth profile.
The chosen experiment (`SINGLE_EXP_ID`) should display a visually prominent oscillation peak.

Adjust `SINGLE_EXP_ID` in the configuration cell above to change the example experiment.

In [ ]:
from simple_pipeline import run_single_experiment
from plotting_utils import plot_simple_comparison

results = run_single_experiment(
    experiment_id=SINGLE_EXP_ID,
    enable_preprocessing=True,
    priors_deviation=0.30,
)

curve_output = FIGURES_DIR / f"single_curve_{SINGLE_EXP_ID}.pdf"
plot_simple_comparison(results, output_path=str(curve_output), save=True)
print(f"Saved: {curve_output}")

## 5. Figure: Model Comparison (Baseline vs Q-Weighted)

Overlays the MAPE histograms of the baseline NF model and the Q-weighted model with dR input.
The baseline appears as grey background bars, the Q-weighted model as coloured foreground bars.

**Sweep commands** (to reproduce from scratch):

```bash
python batch_sweep_runner.py --config sweep_configs/baseline.yaml
python batch_sweep_runner.py --config sweep_configs/qweighted.yaml
```

Then update `QWEIGHTED_ALL_DIR` above to point at the completed Q-weighted batch directory.

In [ ]:
from plotting_utils import plot_model_comparison_histogram, plot_parameter_comparison_grid

baseline_mapes,  _ = extract_mapes_from_batch(BASELINE_ALL_DIR,   use_constraint_mape=True)
qweighted_mapes, _ = extract_mapes_from_batch(QWEIGHTED_ALL_DIR,  use_constraint_mape=True)

output_path = FIGURES_DIR / "model_comparison_histogram.pdf"

plot_model_comparison_histogram(
    baseline_mapes=baseline_mapes,
    model_mapes=qweighted_mapes,
    baseline_label="NF Baseline",
    model_label="NF + Q-Weighted + dR",
    output_path=output_path,
)
print(f"Saved: {output_path}")

# Per-parameter grid comparison
baseline_full = _load_successful(BASELINE_ALL_DIR)
qw_full       = _load_successful(QWEIGHTED_ALL_DIR)

grid_path = FIGURES_DIR / "model_comparison_param_grid.pdf"
plot_parameter_comparison_grid(
    baseline_results=baseline_full,
    model_results=qw_full,
    baseline_label="NF Baseline",
    model_label="NF + Q-Weighted + dR",
    layer_count=1,
    output_path=grid_path,
)
print(f"Saved: {grid_path}")

## 6. Figure: Coverage Plot

The coverage plot shows whether the NF posterior credible intervals are well-calibrated.
For each nominal credible level $\alpha$, it plots the empirical fraction of experiments
where the true parameter falls within the $\alpha$-credible interval.
A perfectly calibrated model lies on the diagonal.

**CLI equivalent**:

```bash
python plot_mape_vs_std.py --batch-dir batch_inference_results/<batch_dir>
```

In [ ]:
from plot_mape_vs_std import plot_coverage, compute_coverage_data

coverage_data = compute_coverage_data(
    str(BASELINE_ALL_DIR / "batch_results.json")
)

coverage_path = FIGURES_DIR / "coverage.pdf"
plot_coverage(coverage_data, output_dir=str(FIGURES_DIR))
print(f"Coverage plot saved to: {FIGURES_DIR}")

## 7. Figure: Evaluation Against Random Guessing

Compares the NF model's MAPE distribution against a random-guessing baseline
that samples parameters uniformly from the prior bounds.

**CLI equivalent**:

```bash
python evaluate_random_guessing.py \
  --batch-dir batch_inference_results/<batch_dir> \
  --num-random-samples 1000 \
  --output-dir figures/
```

In [ ]:
from evaluate_random_guessing import run_random_guessing_evaluation
from plotting_utils import plot_random_guessing_comparison

model_results, random_results = run_random_guessing_evaluation(
    batch_dir=str(BASELINE_ALL_DIR),
    num_random_samples=1000,
)

rg_output = FIGURES_DIR / "random_guessing_comparison.pdf"
plot_random_guessing_comparison(
    model_results=model_results,
    random_results=random_results,
    output_path=str(rg_output),
)
print(f"Saved: {rg_output}")

## 8. Summary: All Figures

| Figure | Output file |
|--------|------------|
| MAPE histograms (baseline) | `figures/baseline/mape_distribution_1layer.pdf` |
| MAPE histograms (q-weighted) | `figures/qweighted/mape_distribution_1layer.pdf` |
| MAPE histograms (mean) | `figures/mean/mape_distribution_1layer.pdf` |
| Model comparison histogram | `figures/model_comparison_histogram.pdf` |
| Model comparison grid | `figures/model_comparison_param_grid.pdf` |
| Coverage | `figures/coverage.pdf` |
| Single curve fit | `figures/single_curve_s000780.pdf` |
| Random guessing | `figures/random_guessing_comparison.pdf` |